In [0]:
display(dbutils.fs.ls("/FileStore/healthcare/raw/"))

In [0]:
%fs ls /FileStore/healthcare/raw/

In [0]:
display(dbutils.fs.ls("/FileStore/tables/healthcare/raw/"))


In [0]:
visits = spark.read.option("header", True).option("inferSchema", True)\
.csv("/FileStore/tables/healthcare/raw/patient_visits.csv")

patients = spark.read.option("header", True).option("inferSchema", True)\
.csv("/FileStore/tables/healthcare/raw/patients.csv")

billing= spark.read.option("header", True).option("inferSchema", True)\
.csv("/FileStore/tables/healthcare/raw/billing.csv")

visits.printSchema()
patients.printSchema()
billing.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

visits_df = visits.withColumn("_ingested_at", current_timestamp())\
.withColumn("_source_file", col("_metadata.file_path"))

patients_df = patients.withColumn("_ingested_at", current_timestamp())\
.withColumn("_source_file", col("_metadata.file_path"))

billing_df = billing.withColumn("_ingested_at", current_timestamp())\
.withColumn("_source_file",col("_metadata.file_path"))

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
visits_df.write.format("delta").mode("overwrite").saveAsTable("bronze.patient_visits")

patients_df.write.format("delta").mode("overwrite").saveAsTable("bronze.patients")

billing_df.write.format("delta").mode("overwrite").saveAsTable("bronze.billing")

In [0]:
%sql
ALTER TABLE bronze.billing
DROP CONSTRAINT valid_consultation_fee;

In [0]:
%sql
ALTER TABLE bronze.billing
ADD CONSTRAINT valid_consultation_fee
CHECK (consultation_fee >= 0);

In [0]:
%sql
ALTER TABLE bronze.patient_visits
ALTER COLUMN visit_id SET NOT NULL;

In [0]:
spark.sql("DESCRIBE HISTORY bronze.patient_visits")

In [0]:
spark.sql("DESCRIBE HISTORY bronze.patient_visits").show()

In [0]:
dbutils.fs.mkdirs("/FileStore/healthcare/new_visits/")

In [0]:
stream_df = spark.readStream \
.format("cloudFiles") \
.option("cloudFiles.format", "csv") \
.option("header", True) \
.option("inferSchema", True) \
.option("cloudFiles.schemaLocation", "/FileStore/schema/patient_visits") \
.load("/FileStore/healthcare/new_visits/")

In [0]:
stream_df.writeStream \
.format("delta") \
.option("checkpointLocation", "/FileStore/checkpoints/patient_visits_stream") \
.outputMode("append") \
.table("bronze.patient_visits_stream")

In [0]:
from pyspark.sql.functions import col, to_date

visits_df = spark.table("bronze.patient_visits")

patients_df = spark.table("bronze.patients")

billing_df = spark.table("bronze.billing")

In [0]:
visits_df.display()

In [0]:
patients_df.display()   

In [0]:
billing_df.display()

In [0]:
visits_df = visits_df \
.withColumnRenamed("_ingested_at","visit_ingested_at") \
.withColumnRenamed("_source_file","visit_source_file")

patients_df = patients_df \
.withColumnRenamed("_ingested_at","patient_ingested_at") \
.withColumnRenamed("_source_file","patient_source_file")

billing_df = billing_df \
.withColumnRenamed("_ingested_at","billing_ingested_at") \
.withColumnRenamed("_source_file","billing_source_file")

In [0]:
visits_df = visits_df.withColumn(
    "visit_date",
    to_date(col("visit_date"), "yyyy-MM-dd")
)

patients_df = patients_df.withColumn(
    "date_of_birth",
    to_date(col("date_of_birth"), "yyyy-MM-dd")
)

In [0]:
from pyspark.sql.functions import floor, months_between

visits_patients_df = visits_df.join(
    patients_df,
    "patient_id"
).withColumn(
    "patient_age_at_visit",
    floor(months_between(col("visit_date"), col("date_of_birth"))/12)
)

In [0]:
billing_df = billing_df.withColumn(
    "total_bill",
    col("consultation_fee")
    + col("medication_cost")
    + col("procedure_cost")
).withColumn(
    "patient_liability",
    col("total_bill") - col("insurance_covered")
)

In [0]:
from pyspark.sql.functions import when

visits_patients_df = visits_patients_df.withColumn(
    "visit_category",
    when(col("duration_days") > 5, "long_stay")
    .when((col("duration_days") >= 2) & (col("duration_days") <= 5), "short_stay")
    .otherwise("day_visit")
)

In [0]:
silver_df = visits_patients_df.join(
    billing_df,
    "visit_id"
)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
silver_df.write \
.format("delta") \
.mode("overwrite") \
.partitionBy("department") \
.saveAsTable("silver.patient_visits")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.patients_scdd (
    patient_id STRING,
    full_name STRING,
    city STRING,
    is_current BOOLEAN,
    effective_start_date DATE,
    effective_end_date DATE
)
USING DELTA;

INSERT INTO silver.patients_scdd
SELECT
    patient_id,
    full_name,
    city,
    true as is_current,
    current_date() as effective_start_date,
    NULL as effective_end_date
FROM silver.patient_visits;

In [0]:
%sql
MERGE INTO silver.patients_scdd target
USING (
    SELECT DISTINCT patient_id, full_name, city
    FROM silver.patient_visits
) source

ON target.patient_id = source.patient_id
AND target.is_current = true

WHEN MATCHED
AND target.city <> source.city

THEN UPDATE SET
    is_current = false,
    effective_end_date = current_date()

WHEN NOT MATCHED

THEN INSERT (
    patient_id,
    full_name,
    city,
    is_current,
    effective_start_date,
    effective_end_date
)

VALUES (
    source.patient_id,
    source.full_name,
    source.city,
    true,
    current_date(),
    NULL
);

In [0]:
%sql
ALTER TABLE silver.patient_visits
SET TBLPROPERTIES (
delta.enableChangeDataFeed = true
);

In [0]:
%sql
SELECT *
FROM table_changes(
'silver.patient_visits',
0
);

In [0]:
%sql
OPTIMIZE silver.patient_visits
ZORDER BY (patient_id, visit_date);

In [0]:
%sql
CREATE OR REPLACE VIEW gold.dept_clinical_summaryy AS

SELECT
    department,
    
    COUNT(*) AS total_visits,

    AVG(duration_days) AS avg_duration_days,

    AVG(patient_liability) AS avg_patient_liability,

    SUM(
        CASE 
            WHEN discharge_status = 'Recovered' 
            THEN 1 
            ELSE 0 
        END
    ) * 1.0 / COUNT(*) AS recovery_rate

FROM silver.patient_visits

GROUP BY department;

In [0]:
%sql
CREATE OR REPLACE VIEW gold.monthly_revenuee AS

SELECT
    
    date_format(visit_date,'yyyy-MM') AS visit_month,

    SUM(total_bill) AS total_revenue,

    SUM(insurance_covered) AS total_insurance_covered

FROM silver.patient_visits

GROUP BY date_format(visit_date,'yyyy-MM')

ORDER BY visit_month;

In [0]:
%sql
SELECT *
FROM gold.dept_clinical_summaryy;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT *
FROM gold.monthly_revenuee;

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
    department,
    visit_date,
    COUNT(*) AS emergency_visits

FROM silver.patient_visits

WHERE visit_type = 'Emergency'

GROUP BY department, visit_date

HAVING COUNT(*) > 10;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.live_admissions
USING DELTA;

In [0]:
from pyspark.sql.functions import col

live_stream_df = spark.readStream \
.format("cloudFiles") \
.option("cloudFiles.format", "csv") \
.option("header", True) \
.option("inferSchema", True) \
.option("cloudFiles.schemaLocation", "/FileStore/schema/live_admissions") \
.load("/FileStore/tables/healthcare/new_visits/")

In [0]:
live_stream_df = live_stream_df \
.withWatermark("visit_date","1 day")

In [0]:
query = live_stream_df.writeStream \
.format("delta") \
.outputMode("append") \
.option("checkpointLocation","/FileStore/checkpoints/live_admissions/") \
.table("gold.live_admissions")

In [0]:
%sql
SELECT COUNT(*) 
FROM gold.live_admissions;

In [0]:
query = live_stream_df.writeStream \
.format("delta") \
.outputMode("append") \
.option("checkpointLocation","/FileStore/checkpoints/live_admissions") \
.table("gold.live_admissions")

In [0]:
%sql
SELECT COUNT(*) 
FROM gold.live_admissions;